# Voice/image cloning for impersonation

# Coding problem, 1.4.a: Classify deepfake/clone call signals

**Skill:** Analyze Risk · **Maps to:** 1.4.A · **Language:** Python 3 (pytest)

## Problem
Build the classifier a call-center security tool would use to flag potentially cloned or deepfake
calls. Given a dictionary of observable signals about an inbound call or video session, return
whether the interaction is `"suspect"` (likely AI-cloned) or `"ok"` (no rule triggered).

## Function contract
Implement `solve(signals: dict) -> str` in the code editor (see the Reference section below).

## Input
A dict with boolean-valued keys describing what was observed. Missing keys are treated as False.
Possible keys: `"unverified_channel"`, `"urgent_money_request"`, `"audio_artifact"`,
`"no_shared_secret"`, `"caller_id_spoofed"`, `"video_face_mismatch"`, `"refused_callback"`.

## Output
The string `"suspect"` if any rule fires, or `"ok"` if none do.

## Examples
```
{"no_shared_secret": True}                                             -> "suspect"
{"audio_artifact": True, "urgent_money_request": True}                -> "suspect"
{"caller_id_spoofed": True, "unverified_channel": True}               -> "suspect"
{"urgent_money_request": True}                                         -> "ok"
{}                                                                     -> "ok"
{"audio_artifact": True}                                               -> "ok"
```

## Classification rules (in order; first match wins)
1. `"no_shared_secret"` is True -> `"suspect"`
2. `"refused_callback"` is True -> `"suspect"`
3. `"audio_artifact"` AND `"urgent_money_request"` -> `"suspect"`
4. `"caller_id_spoofed"` AND `"unverified_channel"` -> `"suspect"`
5. `"video_face_mismatch"` is True -> `"suspect"`
6. `"urgent_money_request"` AND `"unverified_channel"` -> `"suspect"`
7. Otherwise -> `"ok"`

## Constraints
- Deterministic, offline, standard library only.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(signals: dict) -> str
```

### Input (`signals`)
A `dict` with one or more of the following string keys (all optional; missing keys are treated
as absent/False):

| Key | Type | Meaning |
|-----|------|---------|
| `"unverified_channel"` | bool | Call came in on an unexpected or unregistered channel |
| `"urgent_money_request"` | bool | Caller requests money, wire transfer, or credentials urgently |
| `"audio_artifact"` | bool | Robotic cadence, missing background noise, or flat emotional baseline |
| `"no_shared_secret"` | bool | Caller could not supply the pre-agreed code word when asked |
| `"caller_id_spoofed"` | bool | CallerID does not match the authentic registered number |
| `"video_face_mismatch"` | bool | Lighting or sync artifact visible on video |
| `"refused_callback"` | bool | Caller declined when offered a callback on the official number |

### Output (`result`)
A single string: `"suspect"` or `"ok"`.

### Classification rules (applied in order; first match wins)
1. If `"no_shared_secret"` is True -> `"suspect"`.
2. If `"refused_callback"` is True -> `"suspect"`.
3. If `"audio_artifact"` is True AND `"urgent_money_request"` is True -> `"suspect"`.
4. If `"caller_id_spoofed"` is True AND `"unverified_channel"` is True -> `"suspect"`.
5. If `"video_face_mismatch"` is True -> `"suspect"`.
6. If `"urgent_money_request"` is True AND `"unverified_channel"` is True -> `"suspect"`.
7. Otherwise -> `"ok"`.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Missing keys in `signals` are equivalent to `False`.
- Output is always exactly `"suspect"` or `"ok"`.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 1.4.a. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""
from typing import Dict


def solve(signals: Dict[str, bool]) -> str:
    """Classify an inbound call/video session as 'suspect' or 'ok'.

    Args:
        signals: dict of observable boolean indicators (see the Reference section below for keys).
    Returns:
        'suspect' if any classification rule fires, else 'ok'.
    """
    # TODO: apply the ordered rules from the Reference section below / the problem statement above.
    # Missing keys should be treated as False.
    raise NotImplementedError("implement solve() — see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_no_shared_secret_suspect():
    assert impl.solve({"no_shared_secret": True}) == "suspect"


def test_refused_callback_suspect():
    assert impl.solve({"refused_callback": True}) == "suspect"


def test_audio_plus_money_suspect():
    assert impl.solve({"audio_artifact": True, "urgent_money_request": True}) == "suspect"


def test_spoofed_plus_unverified_suspect():
    assert impl.solve({"caller_id_spoofed": True, "unverified_channel": True}) == "suspect"


def test_video_face_mismatch_suspect():
    assert impl.solve({"video_face_mismatch": True}) == "suspect"


def test_money_plus_unverified_suspect():
    assert impl.solve({"urgent_money_request": True, "unverified_channel": True}) == "suspect"


def test_empty_ok():
    assert impl.solve({}) == "ok"


def test_single_money_ok():
    # urgent money alone (no second signal) is NOT enough under rules 3 or 6
    assert impl.solve({"urgent_money_request": True}) == "ok"


def test_audio_only_ok():
    # audio artifact alone does not trigger without money request
    assert impl.solve({"audio_artifact": True}) == "ok"


# ---- hidden / edge cases -------------------------------------------------
def test_missing_keys_treated_false():
    # caller_id_spoofed True but unverified_channel absent -> rule 4 does not fire
    assert impl.solve({"caller_id_spoofed": True}) == "ok"


def test_all_false_ok():
    signals = {k: False for k in [
        "unverified_channel", "urgent_money_request", "audio_artifact",
        "no_shared_secret", "caller_id_spoofed", "video_face_mismatch", "refused_callback",
    ]}
    assert impl.solve(signals) == "ok"


def test_rule1_takes_priority():
    # even if other signals are also set, rule 1 should still fire first
    assert impl.solve({
        "no_shared_secret": True,
        "audio_artifact": True,
        "urgent_money_request": True,
        "refused_callback": True,
    }) == "suspect"


def test_unverified_channel_alone_ok():
    assert impl.solve({"unverified_channel": True}) == "ok"


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
